# Laguna XS.2 — Final Causal Expert Surgery
## AWS g7e.2xlarge · 1× RTX PRO 6000 Blackwell 96GB · 8 vCPU · 64 GiB RAM

This is the **direct-BF16 PyTorch/Transformers research path**.

No vLLM. No INT4 conversion. No tensor parallelism. No CPU model offload.

The notebook loads the official `poolside/Laguna-XS.2` BF16 checkpoint and performs:

1. hardware/storage/memory preflight;
2. direct BF16 model loading;
3. exact fixed-routing causal layer ablation;
4. hierarchical expert-group search;
5. exact individual expert validation;
6. bootstrap confidence intervals;
7. renormalization robustness;
8. routing-frequency vs causal-rank comparison;
9. optional expert-pair interactions;
10. optional **full-rank training of only selected expert slices** through a compact external trainable expert bank.

### Hardware target

- GPU: RTX PRO 6000 Blackwell 96GB (~89 GiB binary)
- CPU: 8 vCPU
- system RAM: 64 GiB
- local SSD/NVMe: ideally 100GB+ free

### Model facts validated in-notebook

Expected Laguna XS.2 configuration:

- 40 transformer layers
- 39 sparse MoE layers
- hidden size 2048
- 256 routed experts per sparse layer
- top-8 routing
- expert width 512
- fused expert tensors:
  - `gate_up_proj[256, 1024, 2048]`
  - `down_proj[256, 2048, 512]`

### Core causal score

\[
S(E)=\Delta L_{target}(E)-\lambda\max(0,\Delta L_{control}(E))
\]

Routing frequency is measured only **after** causal selection.

> **Final hardening:** explicitly registers Transformers' native Laguna
> checkpoint-conversion mapping before BF16 loading and validates all critical
> MoE keys before causal experiments begin.


> **v2 scoring fix:** teacher-forced references now follow Laguna's exact
> no-thinking assistant syntax (`</think>\n<answer>`), and a full-logit
> equivalence check runs before causal search.


> **v3 matched-baseline extension:** after the corrected causal atlas, this
> notebook can run a matched one-expert adaptation experiment comparing the
> strongest causal expert against the most-routed expert and random controls on
> separate selection/train/held-out splits.


## 1 — Install research dependencies

In [1]:
# Keep the environment's CUDA-enabled PyTorch. Do not reinstall torch.
%pip -q install -U \
  "transformers==5.14.1" \
  "accelerate>=1.10.0" \
  "huggingface_hub>=0.35.0" \
  safetensors pandas numpy psutil tqdm matplotlib scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2 — Runtime tuning for AWS g7e.2xlarge (8 vCPU / 64 GiB RAM)

In [2]:
import os

# ============================================================
# g7e.2xlarge tuning
# 8 vCPU / 64 GiB RAM / RTX PRO 6000 96GB
# ============================================================

# Leave ~2 CPU threads free for Python, I/O, HF download, etc.
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

# Tokenization can use parallel CPU work, but don't aggressively oversubscribe.
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# Lower allocator arena count for 64 GiB host RAM.
os.environ["MALLOC_ARENA_MAX"] = "4"

# ============================================================
# Hugging Face checkpoint loading
# ============================================================

# Parallel loading is still useful, but 8 simultaneous ~5 GB shards
# is too aggressive for only 64 GiB system RAM.
os.environ["HF_ENABLE_PARALLEL_LOADING"] = "true"
os.environ["HF_PARALLEL_LOADING_WORKERS"] = "2"

# Xet download concurrency: enough to saturate networking without
# hammering all 8 CPU cores / RAM.
os.environ["HF_XET_NUM_CONCURRENT_RANGE_GETS"] = "8"

# Avoid using host storage/RAM for the optional Xet chunk cache.
os.environ["HF_XET_CHUNK_CACHE_SIZE_BYTES"] = "0"

# ============================================================
# CUDA / RTX PRO 6000
# ============================================================

os.environ["CUDA_MODULE_LOADING"] = "LAZY"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,max_split_size_mb:512"
)

print("Runtime configured for g7e.2xlarge.")

Runtime configured for g7e.2xlarge.


In [3]:
import torch

torch.set_num_threads(6)

try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

## 3 — Hardware and storage preflight

In [4]:
import os, shutil, platform
from pathlib import Path
import psutil


ram = psutil.virtual_memory()

print("=== Host ===")
print("Python:", platform.python_version())
print("Logical CPUs:", os.cpu_count())
print(f"RAM total:     {ram.total/2**30:.2f} GiB")
print(f"RAM available: {ram.available/2**30:.2f} GiB")

print("\n=== CUDA ===")
print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not visible.")

if torch.cuda.device_count() != 1:
    raise RuntimeError(f"Expected exactly one GPU; found {torch.cuda.device_count()}.")

props = torch.cuda.get_device_properties(0)
print("GPU:", props.name)
print(f"VRAM: {props.total_memory/2**30:.2f} GiB")
print("Compute capability:", torch.cuda.get_device_capability(0))

if props.total_memory / 2**30 < 88:
    raise RuntimeError("Need a 96GB-class GPU (~89 GiB binary).")

if (os.cpu_count() or 0) < 8:
    print("WARNING: fewer than 8 logical CPUs detected.")

if ram.total / 2**30 < 58:
    print("WARNING: less than ~64 GiB-class RAM detected.")

roots = [
    Path("/home/ec2-user/workspace"),
    Path("/workspace"),
    Path("/mnt/data"),
    Path("/root"),
    Path("/tmp"),
    Path.cwd(),
]

choices = []
seen = set()
for p in roots:
    try:
        if not p.exists() or not os.access(p, os.W_OK):
            continue
        dev = os.stat(p).st_dev
        if dev in seen:
            continue
        seen.add(dev)
        usage = shutil.disk_usage(p)
        choices.append((usage.free, p, usage))
    except OSError:
        pass

if not choices:
    raise RuntimeError("No writable filesystem found.")

_, WORK_ROOT, disk = max(choices, key=lambda x: x[0])

print("\n=== Storage ===")
print("Work root:", WORK_ROOT)
print(f"Disk free: {disk.free/2**30:.2f} GiB")
print("Hardware preflight: PASS")

=== Host ===
Python: 3.10.12
Logical CPUs: 8
RAM total:     62.27 GiB
RAM available: 58.59 GiB

=== CUDA ===
Torch: 2.13.0+cu130
CUDA build: 13.0
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 94.97 GiB
Compute capability: (12, 0)

=== Storage ===
Work root: /home/ec2-user/workspace
Disk free: 918.38 GiB
Hardware preflight: PASS


## 4 — Resolve the official BF16 checkpoint

In [5]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "poolside/Laguna-XS.2"

default_model_path = (
    Path("/home/ec2-user/workspace/models/Laguna-XS.2")
    if Path("/home/ec2-user/workspace").exists()
    else WORK_ROOT / "models" / "Laguna-XS.2"
)

MODEL_PATH = Path(
    os.environ.get("LAGUNA_BF16_PATH", str(default_model_path))
).expanduser().resolve()

EXPECTED_SHARDS = [
    f"model-{i:05d}-of-00014.safetensors"
    for i in range(1, 15)
]

complete = (
    (MODEL_PATH / "config.json").exists()
    and all((MODEL_PATH / x).exists() for x in EXPECTED_SHARDS)
)

if not complete:
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    free_gib = shutil.disk_usage(MODEL_PATH).free / 2**30

    if free_gib < 85:
        raise RuntimeError(
            f"Need ~85 GiB free for a fresh BF16 download; found {free_gib:.1f} GiB. "
            "Mount the checkpoint elsewhere and set LAGUNA_BF16_PATH."
        )

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_PATH),
        allow_patterns=[
            "*.safetensors", "*.json", "*.py", "*.jinja",
            "LICENSE*", "README*",
        ],
        max_workers=2,
    )

missing = [x for x in EXPECTED_SHARDS if not (MODEL_PATH / x).exists()]
if missing:
    raise RuntimeError(f"Incomplete BF16 checkpoint; missing: {missing}")

sizes = [(x, (MODEL_PATH / x).stat().st_size) for x in EXPECTED_SHARDS]

print("MODEL_PATH:", MODEL_PATH)
print(f"14-shard tensor payload: {sum(n for _, n in sizes)/1e9:.3f} GB")
print(f"First shard: {sizes[0][1]/1e9:.3f} GB")
print(f"Last shard:  {sizes[-1][1]/1e9:.3f} GB")
print("Checkpoint verification: PASS")

MODEL_PATH: /home/ec2-user/workspace/models/Laguna-XS.2
14-shard tensor payload: 66.889 GB
First shard: 5.120 GB
Last shard:  0.336 GB
Checkpoint verification: PASS


/home/ec2-user/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 5 — Explicit Laguna checkpoint-conversion registration

Transformers' dynamic weight loader can consolidate per-expert checkpoint
tensors into Laguna's fused 3D expert tensors. For `trust_remote_code` models,
native mappings may be skipped unless explicitly registered.

Register the built-in `laguna` mapping before the 66.9 GB BF16 load. This is
idempotent and prevents the exact class of per-expert/fused-key mismatch seen
with the quantized checkpoint.

In [6]:
from transformers.conversion_mapping import (
    get_checkpoint_conversion_mapping,
    register_checkpoint_conversion_mapping,
    USER_REGISTERED_MAPPINGS,
)

laguna_mapping = get_checkpoint_conversion_mapping("laguna")

if laguna_mapping is None:
    raise RuntimeError(
        "This Transformers installation does not expose the native Laguna "
        "checkpoint conversion mapping."
    )

register_checkpoint_conversion_mapping(
    "laguna",
    laguna_mapping,
    overwrite=True,
)

if "laguna" not in USER_REGISTERED_MAPPINGS:
    raise RuntimeError("Laguna conversion mapping registration failed.")

print("Laguna checkpoint conversion mapping: REGISTERED")
print("Conversion operations:", len(laguna_mapping))

Laguna checkpoint conversion mapping: REGISTERED
Conversion operations: 4


## 6 — Load BF16 directly onto the RTX PRO 6000

In [7]:
import gc, time, torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(min(6, os.cpu_count() or 6))
try:
    torch.set_num_interop_threads(2)
except RuntimeError:
    pass

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("Transformers:", transformers.__version__)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    fix_mistral_regex=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.time()

model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    device_map={"": 0},
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
    attn_implementation="sdpa",
    output_loading_info=True,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad_(False)

missing = list(loading_info.get("missing_keys", []))
unexpected = list(loading_info.get("unexpected_keys", []))
mismatched = list(loading_info.get("mismatched_keys", []))

critical = [
    k for k in (missing + unexpected)
    if ".mlp.experts" in k or ".mlp.gate" in k or "e_score_correction_bias" in k
]

if critical or mismatched:
    raise RuntimeError(
        "Critical checkpoint mismatch.\n"
        f"critical sample: {critical[:12]}\n"
        f"mismatched sample: {mismatched[:12]}"
    )

del loading_info
gc.collect()
torch.cuda.synchronize()

free_b, total_b = torch.cuda.mem_get_info()

print(f"Loaded in {(time.time()-t0)/60:.2f} min")
print(f"GPU allocated: {torch.cuda.memory_allocated()/2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved()/2**30:.2f} GiB")
print(f"GPU peak:      {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free:   {free_b/2**30:.2f} GiB")
print(f"Host RAM available: {psutil.virtual_memory().available/2**30:.2f} GiB")

print("BF16 load: PASS")

Transformers: 5.14.1


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 639/639 [05:47<00:00,  1.84it/s]


Loaded in 5.85 min
GPU allocated: 62.29 GiB
GPU reserved:  64.81 GiB
GPU peak:      63.29 GiB
Driver free:   29.61 GiB
Host RAM available: 58.27 GiB
BF16 load: PASS


## 6 — Validate exact Laguna MoE structure

In [8]:
cfg = model.config

SPARSE_LAYERS = []
for idx, layer in enumerate(model.model.layers):
    mlp = getattr(layer, "mlp", None)
    if (
        mlp is not None
        and hasattr(mlp, "gate")
        and hasattr(mlp, "experts")
        and hasattr(mlp.experts, "gate_up_proj")
        and hasattr(mlp.experts, "down_proj")
    ):
        SPARSE_LAYERS.append(idx)

print("hidden_size:", cfg.hidden_size)
print("layers:", cfg.num_hidden_layers)
print("experts:", cfg.num_experts)
print("top_k:", cfg.num_experts_per_tok)
print("expert width:", cfg.moe_intermediate_size)
print("sparse layers:", SPARSE_LAYERS)

assert cfg.hidden_size == 2048
assert cfg.num_hidden_layers == 40
assert cfg.num_experts == 256
assert cfg.num_experts_per_tok == 8
assert cfg.moe_intermediate_size == 512
assert len(SPARSE_LAYERS) == 39

sample = model.model.layers[SPARSE_LAYERS[0]].mlp
print("gate_up_proj:", tuple(sample.experts.gate_up_proj.shape), sample.experts.gate_up_proj.dtype)
print("down_proj:   ", tuple(sample.experts.down_proj.shape), sample.experts.down_proj.dtype)
print("router:", tuple(sample.gate.weight.shape), sample.gate.weight.dtype)

assert tuple(sample.experts.gate_up_proj.shape) == (256, 1024, 2048)
assert tuple(sample.experts.down_proj.shape) == (256, 2048, 512)

params_per_expert = (
    sample.experts.gate_up_proj[0].numel()
    + sample.experts.down_proj[0].numel()
)

print(f"params/expert: {params_per_expert:,} ({params_per_expert/1e6:.3f}M)")
print("MoE structure: PASS")

hidden_size: 2048
layers: 40
experts: 256
top_k: 8
expert width: 512
sparse layers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
gate_up_proj: (256, 1024, 2048) torch.bfloat16
down_proj:    (256, 2048, 512) torch.bfloat16
router: (256, 2048) torch.bfloat16
params/expert: 3,145,728 (3.146M)
MoE structure: PASS


## 7 — Short forward smoke test

In [9]:
@torch.inference_mode()
def smoke_forward(text):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    )
    enc = {k: v.to("cuda:0", non_blocking=True) for k, v in enc.items()}

    return model(
        **enc,
        use_cache=False,
        logits_to_keep=1,
        return_dict=True,
    ).logits

torch.cuda.reset_peak_memory_stats()

logits = smoke_forward(
    "Explain briefly why min-width: 0 can matter inside a CSS flex container."
)

torch.cuda.synchronize()
free_b, _ = torch.cuda.mem_get_info()

print("logits shape:", tuple(logits.shape))
print(f"Peak allocation: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"Driver free: {free_b/2**30:.2f} GiB")

del logits
torch.cuda.empty_cache()

logits shape: (1, 1, 100352)
Peak allocation: 62.33 GiB
Driver free: 29.48 GiB


# Phase A — Target/control evaluation set

The bundled examples are only a pipeline validation set.

For publication-quality work, replace them with:
- 50–200+ target-selection examples,
- 50–200+ matched controls,
- a separate held-out evaluation set.

In [10]:
import pandas as pd

EVAL_ROWS = [
    {"kind":"target","prompt":"In CSS flexbox, what declaration lets a flex item shrink below its intrinsic content width?","reference":"min-width: 0;"},
    {"kind":"target","prompt":"What CSS declaration establishes a flex formatting context?","reference":"display: flex;"},
    {"kind":"target","prompt":"Which React hook stores local component state?","reference":"useState"},
    {"kind":"target","prompt":"What CSS property controls horizontal overflow?","reference":"overflow-x"},
    {"kind":"target","prompt":"Which CSS property controls stacking order for positioned elements?","reference":"z-index"},
    {"kind":"target","prompt":"Which declaration commonly centers flex children along the main axis?","reference":"justify-content: center;"},
    {"kind":"target","prompt":"In React, which prop gives a stable identity to list items?","reference":"key"},
    {"kind":"target","prompt":"Which CSS property sets spacing between grid or flex children without margins?","reference":"gap"},
    {"kind":"target","prompt":"Which CSS property includes padding and border inside declared width?","reference":"box-sizing"},
    {"kind":"target","prompt":"Which browser API observes element size changes?","reference":"ResizeObserver"},
    {"kind":"target","prompt":"Which React hook runs side effects after rendering?","reference":"useEffect"},
    {"kind":"target","prompt":"Which CSS property defines grid columns?","reference":"grid-template-columns"},

    {"kind":"control","prompt":"Which Python keyword yields a value from a generator?","reference":"yield"},
    {"kind":"control","prompt":"Which traversal finds shortest paths in an unweighted graph?","reference":"BFS"},
    {"kind":"control","prompt":"Which Java keyword declares class inheritance?","reference":"extends"},
    {"kind":"control","prompt":"Which SQL keyword removes duplicate SELECT rows?","reference":"DISTINCT"},
    {"kind":"control","prompt":"Which C++ smart pointer represents exclusive ownership?","reference":"std::unique_ptr"},
    {"kind":"control","prompt":"Which asymptotic notation describes an upper bound?","reference":"Big O"},
    {"kind":"control","prompt":"Which Python container provides average O(1) membership lookup for hashable values?","reference":"set"},
    {"kind":"control","prompt":"Which data structure is first-in first-out?","reference":"queue"},
    {"kind":"control","prompt":"Which SQL clause filters groups after aggregation?","reference":"HAVING"},
    {"kind":"control","prompt":"Which Java interface defines natural ordering?","reference":"Comparable"},
    {"kind":"control","prompt":"Which C++ keyword prevents modification through that name?","reference":"const"},
    {"kind":"control","prompt":"What mathematical operation is the inverse of exponentiation for solving an exponent?","reference":"logarithm"},
]

eval_df = pd.DataFrame(EVAL_ROWS)
display(eval_df.groupby("kind").size().rename("count"))

kind
control    12
target     12
Name: count, dtype: int64

## 8 — Build one reusable aligned GPU scoring batch

In [11]:
def chat_prefix_text(prompt):
    messages = [{"role":"user","content":prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def parse_case(prompt, reference):
    prefix_text = chat_prefix_text(prompt)
    prefix_ids = tokenizer.encode(prefix_text, add_special_tokens=False)
    # Laguna's no-thinking assistant template begins content after
    # </think> with a newline. Teacher forcing must match that exact syntax.
    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    start = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise RuntimeError(f"Could not identify answer boundary for {prompt!r}")

    return full_ids, start

def build_scoring_batch(df):
    parsed = [parse_case(r.prompt, r.reference) for r in df.itertuples(index=False)]
    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros((batch_size, seq_len), dtype=torch.long)
    targets = torch.full((batch_size, max_ref), -100, dtype=torch.long)

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1
        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

SCORE_BATCH = build_scoring_batch(eval_df)

print("Batch:", SCORE_BATCH["input_ids"].shape)
print("Reference positions:", len(SCORE_BATCH["pred_positions"]))

Batch: torch.Size([24, 74])
Reference positions: 7


## 9 — Cached teacher-forced reference NLL

In [12]:
import numpy as np
import torch.nn.functional as F

@torch.inference_mode()
def score_cached_batch():
    out = model(
        input_ids=SCORE_BATCH["input_ids"],
        attention_mask=SCORE_BATCH["attention_mask"],
        position_ids=SCORE_BATCH["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=SCORE_BATCH["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = SCORE_BATCH["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)
    per_example = (losses * valid).sum(-1) / valid.sum(-1).clamp_min(1)
    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example
    return result

BASE_NLL = score_cached_batch()

target_mask = eval_df["kind"].values == "target"
control_mask = eval_df["kind"].values == "control"

print("Baseline target NLL: ", float(BASE_NLL[target_mask].mean()))
print("Baseline control NLL:", float(BASE_NLL[control_mask].mean()))

Baseline target NLL:  5.3066630363464355
Baseline control NLL: 5.52145528793335


### Scoring sanity check — required before causal search

Two things are verified here:

1. the no-thinking assistant prefix is followed by a **newline** before the
   reference answer, matching Laguna's chat template;
2. `logits_to_keep=<tensor of positions>` produces exactly the same selected
   logits as running the full LM head and slicing those positions afterward.

This catches malformed teacher-forcing syntax and off-by-one logit-position
errors before any causal ranking is computed.

In [13]:
# Show the exact generation boundary.
_demo_prefix = chat_prefix_text(eval_df.iloc[0]["prompt"])
print("Prefix tail:", repr(_demo_prefix[-60:]))
print("Teacher-forced boundary:", repr((_demo_prefix + "\n" + eval_df.iloc[0]["reference"])[-90:]))

if not _demo_prefix.endswith("</think>"):
    print(
        "WARNING: generation prefix no longer ends in </think>; "
        "inspect the current Laguna chat template."
    )

# Per-example baseline NLLs are useful for catching malformed prompts.
baseline_detail = eval_df[["kind", "prompt", "reference"]].copy()
baseline_detail["baseline_nll"] = BASE_NLL

print("\nBaseline NLL by kind:")
display(
    baseline_detail.groupby("kind")["baseline_nll"].agg(
        ["mean", "median", "min", "max"]
    )
)

display(
    baseline_detail.sort_values("baseline_nll", ascending=False).head(8)
)

# Verify selective-logit scoring against a full-logit forward for one example.
_b = 0
_ids = SCORE_BATCH["input_ids"][_b:_b+1]
_mask = SCORE_BATCH["attention_mask"][_b:_b+1]
_pos = SCORE_BATCH["position_ids"][_b:_b+1]
_keep = SCORE_BATCH["pred_positions"]

with torch.inference_mode():
    _sel = model(
        input_ids=_ids,
        attention_mask=_mask,
        position_ids=_pos,
        use_cache=False,
        logits_to_keep=_keep,
        return_dict=True,
    ).logits.float()

    _full = model(
        input_ids=_ids,
        attention_mask=_mask,
        position_ids=_pos,
        use_cache=False,
        logits_to_keep=0,
        return_dict=True,
    ).logits[:, _keep, :].float()

_max_logit_diff = float((_sel - _full).abs().max().item())
print("max |selective logits - full sliced logits|:", _max_logit_diff)

if _max_logit_diff > 1e-4:
    raise RuntimeError(
        "Selective-logit scorer does not match full-logit scoring. "
        "Stop before causal search."
    )

del _sel, _full
torch.cuda.empty_cache()

print("Scoring implementation sanity: PASS")


Prefix tail: 'ow its intrinsic content width?\n</user>\n<assistant>\n</think>'
Teacher-forced boundary: ' item shrink below its intrinsic content width?\n</user>\n<assistant>\n</think>\nmin-width: 0;'

Baseline NLL by kind:


,mean,median,min,max
kind,,,,
control,5.521455,5.705105,0.709383,9.690764
target,5.306663,4.874743,2.984896,8.728827


,kind,prompt,reference,baseline_nll
12,control,Which Python keyword yields a value from a gen...,yield,9.690764
6,target,"In React, which prop gives a stable identity t...",key,8.728827
2,target,Which React hook stores local component state?,useState,8.170768
14,control,Which Java keyword declares class inheritance?,extends,7.953543
7,target,Which CSS property sets spacing between grid o...,gap,7.918708
18,control,Which Python container provides average O(1) m...,set,7.882426
21,control,Which Java interface defines natural ordering?,Comparable,7.820670
22,control,Which C++ keyword prevents modification throug...,const,7.130175


max |selective logits - full sliced logits|: 0.0
Scoring implementation sanity: PASS


# Phase B — Fixed-routing causal intervention

The router is called normally first.

The patch changes only returned `routing_weights`; `selected_experts` remains
unchanged, so no replacement expert is allowed to enter the top-k.

In [14]:
from contextlib import contextmanager, ExitStack
import types

def get_sparse_mlp(layer_idx):
    layer_idx = int(layer_idx)
    if layer_idx not in SPARSE_LAYERS:
        raise ValueError(f"Layer {layer_idx} is not sparse.")
    return model.model.layers[layer_idx].mlp

@contextmanager
def gate_intervention(layer_idx, expert_ids=None, zero_all_routed=False, renormalize=False):
    mlp = get_sparse_mlp(layer_idx)
    gate = mlp.gate
    original_forward = gate.forward
    expert_ids = [] if expert_ids is None else [int(x) for x in expert_ids]

    def patched_forward(self, hidden_states):
        router_logits, routing_weights, selected_experts = original_forward(hidden_states)

        if zero_all_routed:
            routing_weights = torch.zeros_like(routing_weights)

        elif expert_ids:
            ids = torch.tensor(
                expert_ids,
                device=selected_experts.device,
                dtype=selected_experts.dtype,
            )
            keep = ~torch.isin(selected_experts, ids)
            routing_weights = routing_weights * keep.to(routing_weights.dtype)

            if renormalize:
                denom = routing_weights.sum(-1, keepdim=True)
                routing_weights = torch.where(
                    denom > 0,
                    routing_weights / denom.clamp_min(1e-12),
                    routing_weights,
                )

        return router_logits, routing_weights, selected_experts

    gate.forward = types.MethodType(patched_forward, gate)
    try:
        yield
    finally:
        gate.forward = original_forward

## 10 — Intervention integrity check

In [15]:
test_layer = SPARSE_LAYERS[len(SPARSE_LAYERS)//2]

with gate_intervention(test_layer, zero_all_routed=True):
    test_nll = score_cached_batch()

delta = test_nll - BASE_NLL

print("Test layer:", test_layer)
print("Mean |ΔNLL|:", float(np.abs(delta).mean()))

if np.abs(delta).mean() < 1e-7:
    raise RuntimeError("Intervention produced no meaningful change.")

print("Fixed-routing intervention: PASS")

Test layer: 20
Mean |ΔNLL|: 0.4701980650424957
Fixed-routing intervention: PASS


## 11 — Causal layer sweep

In [16]:
from tqdm.auto import tqdm
import time

CONTROL_PENALTY = 0.75

def summarize_delta(ablated):
    delta = np.asarray(ablated) - BASE_NLL
    td = float(delta[target_mask].mean())
    cd = float(delta[control_mask].mean())

    return {
        "target_delta_nll": td,
        "control_delta_nll": cd,
        "causal_specificity": td - CONTROL_PENALTY * max(cd, 0.0),
        "per_example_delta": delta,
    }

RESULTS = WORK_ROOT / "laguna_xs2_causal_surgery_results"
RESULTS.mkdir(parents=True, exist_ok=True)

layer_rows = []
t0 = time.time()

for layer_idx in tqdm(SPARSE_LAYERS, desc="39-layer causal sweep"):
    with gate_intervention(layer_idx, zero_all_routed=True):
        nll = score_cached_batch()

    m = summarize_delta(nll)
    layer_rows.append({
        "layer": int(layer_idx),
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
    })

layer_df = pd.DataFrame(layer_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

layer_df.to_csv(RESULTS / "layer_causal_scores.csv", index=False)

print(f"Layer sweep: {(time.time()-t0)/60:.2f} min")
display(layer_df.head(15))

39-layer causal sweep: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:08<00:00,  4.52it/s]

Layer sweep: 0.14 min


,layer,target_delta_nll,control_delta_nll,causal_specificity
0,36,1.379729,-0.178304,1.379729
1,27,0.376631,0.093129,0.306784
2,38,0.288470,-0.089592,0.288470
3,33,0.263263,0.002165,0.261639
4,31,0.068031,-0.156100,0.068031
5,34,0.055333,-0.067383,0.055333
6,25,0.377989,0.482786,0.015900
7,8,0.037709,0.059897,-0.007214
8,22,-0.020884,-0.243931,-0.020884
9,13,-0.040551,-0.079879,-0.040551


## 12 — Hierarchical expert-group search

In [17]:
def intervention_score(layer_idx, expert_ids, renormalize=False):
    with gate_intervention(
        layer_idx,
        expert_ids=expert_ids,
        renormalize=renormalize,
    ):
        nll = score_cached_batch()

    return summarize_delta(nll)

def hierarchical_expert_search(
    layer_idx,
    seed=17,
    initial_group_size=32,
    beam_width=3,
):
    rng = np.random.default_rng(seed)
    order = rng.permutation(cfg.num_experts).tolist()

    frontier = [
        order[i:i+initial_group_size]
        for i in range(0, len(order), initial_group_size)
    ]

    history = []
    level = 0

    while frontier:
        current = []

        for group in tqdm(
            frontier,
            desc=f"L{layer_idx} level {level}",
            leave=False,
        ):
            m = intervention_score(layer_idx, group)

            rec = {
                "layer": int(layer_idx),
                "seed": int(seed),
                "level": int(level),
                "group_size": len(group),
                "experts": list(map(int, group)),
                "target_delta_nll": m["target_delta_nll"],
                "control_delta_nll": m["control_delta_nll"],
                "causal_specificity": m["causal_specificity"],
            }

            history.append(rec)
            current.append(rec)

        current.sort(key=lambda x: x["causal_specificity"], reverse=True)
        keep = current[:beam_width]

        if all(x["group_size"] == 1 for x in keep):
            break

        nxt = []
        for rec in keep:
            g = rec["experts"]
            if len(g) == 1:
                nxt.append(g)
            else:
                mid = len(g)//2
                nxt.extend([g[:mid], g[mid:]])

        frontier = [x for x in nxt if x]
        level += 1

    hist = pd.DataFrame(history)
    leaf_size = hist["group_size"].min()
    leaves = hist[hist.group_size == leaf_size].sort_values(
        "causal_specificity",
        ascending=False,
    )

    return hist, leaves

## 13 — Search top causal layers with two expert-order seeds

In [18]:
TOP_LAYERS = 4
SEARCH_SEEDS = [17, 53]
INITIAL_GROUP_SIZE = 32
BEAM_WIDTH = 3

candidate_layers = layer_df.head(TOP_LAYERS)["layer"].astype(int).tolist()
print("Candidate layers:", candidate_layers)

histories = []
leaf_frames = []

for layer_idx in candidate_layers:
    for seed in SEARCH_SEEDS:
        hist, leaves = hierarchical_expert_search(
            layer_idx,
            seed=seed,
            initial_group_size=INITIAL_GROUP_SIZE,
            beam_width=BEAM_WIDTH,
        )
        histories.append(hist)
        leaf_frames.append(leaves)

group_history = pd.concat(histories, ignore_index=True)
leaf_df = pd.concat(leaf_frames, ignore_index=True)

group_history.to_json(
    RESULTS / "hierarchical_group_history.json",
    orient="records",
    indent=2,
)

display(leaf_df.head(30))

Candidate layers: [36, 27, 38, 33]


,layer,seed,level,group_size,experts,target_delta_nll,control_delta_nll,causal_specificity
0,36,17,5,1,[229],1.285752,-0.031845,1.285752
1,36,17,5,1,[252],0.111675,0.010622,0.103708
2,36,17,5,1,[250],0.018760,0.015421,0.007195
3,36,17,5,1,[204],0.000000,-0.000015,0.000000
4,36,17,5,1,[172],-0.000878,0.001454,-0.001969
5,36,17,5,1,[149],-0.002534,0.004454,-0.005875
6,36,53,5,1,[229],1.285752,-0.031845,1.285752
7,36,53,5,1,[252],0.111675,0.010622,0.103708
8,36,53,5,1,[37],0.010175,-0.005212,0.010175
9,36,53,5,1,[122],0.000434,-0.012858,0.000434


## 14 — Exact individual expert validation

In [19]:
leaf_pairs = sorted({
    (int(r.layer), int(e))
    for r in leaf_df.itertuples(index=False)
    for e in r.experts
})

print("Unique leaf candidates:", len(leaf_pairs))

individual_rows = []

for layer_idx, expert_id in tqdm(leaf_pairs, desc="Individual validation"):
    m = intervention_score(layer_idx, [expert_id])

    individual_rows.append({
        "layer": layer_idx,
        "expert": expert_id,
        "target_delta_nll": m["target_delta_nll"],
        "control_delta_nll": m["control_delta_nll"],
        "causal_specificity": m["causal_specificity"],
        "per_example_delta": m["per_example_delta"].tolist(),
    })

individual_df = pd.DataFrame(individual_rows).sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

individual_df.drop(columns=["per_example_delta"]).to_csv(
    RESULTS / "individual_causal_experts.csv",
    index=False,
)

display(individual_df.head(25))

Unique leaf candidates: 39


Individual validation: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:08<00:00,  4.59it/s]


,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,per_example_delta
0,36,229,1.285752,-0.031845,1.285752,"[1.2339627742767334, 1.6525044441223145, -0.09..."
1,38,60,0.191670,0.002457,0.189827,"[0.3744649887084961, 0.48745298385620117, 0.08..."
2,36,252,0.111675,0.010622,0.103708,"[0.018002748489379883, 0.07581663131713867, 0...."
3,27,146,0.232747,0.180549,0.097336,"[0.11967086791992188, -0.04862785339355469, 0...."
4,27,40,0.092108,-0.004402,0.092108,"[0.04875349998474121, 0.12630462646484375, -0...."
5,33,1,0.090867,-0.025969,0.090867,"[-0.2565634250640869, 0.25287938117980957, -0...."
6,38,210,0.068980,0.000004,0.068977,"[-0.016318321228027344, -0.08301615715026855, ..."
7,33,251,0.057142,-0.003049,0.057142,"[0.006606578826904297, -0.05129814147949219, 0..."
8,33,169,0.042570,-0.003009,0.042570,"[0.0037183761596679688, 0.021992921829223633, ..."
9,27,69,0.033293,-0.045648,0.033293,"[-0.04130697250366211, -0.008118391036987305, ..."


## 15 — Bootstrap confidence intervals

In [20]:
def bootstrap_specificity(delta, n_boot=5000, seed=123):
    rng = np.random.default_rng(seed)
    d = np.asarray(delta, dtype=np.float64)

    t = d[target_mask]
    c = d[control_mask]

    vals = np.empty(n_boot, dtype=np.float64)

    for i in range(n_boot):
        tb = rng.choice(t, size=len(t), replace=True).mean()
        cb = rng.choice(c, size=len(c), replace=True).mean()
        vals[i] = tb - CONTROL_PENALTY * max(cb, 0.0)

    return {
        "ci_2.5": float(np.quantile(vals, 0.025)),
        "ci_97.5": float(np.quantile(vals, 0.975)),
        "p_positive": float((vals > 0).mean()),
    }

boot_rows = []

for r in individual_df.itertuples(index=False):
    boot_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        **bootstrap_specificity(r.per_example_delta),
    })

boot_df = pd.DataFrame(boot_rows)

final_df = individual_df.merge(
    boot_df,
    on=["layer","expert"],
    how="left",
).sort_values(
    ["p_positive","causal_specificity"],
    ascending=False,
).reset_index(drop=True)

display(final_df.head(25))

,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,per_example_delta,ci_2.5,ci_97.5,p_positive
0,36,229,1.285752,-0.031845,1.285752,"[1.2339627742767334, 1.6525044441223145, -0.09...",0.451026,2.197427,1.0000
1,38,60,0.191670,0.002457,0.189827,"[0.3744649887084961, 0.48745298385620117, 0.08...",0.017480,0.355775,0.9858
2,38,210,0.068980,0.000004,0.068977,"[-0.016318321228027344, -0.08301615715026855, ...",0.003809,0.150821,0.9810
3,33,251,0.057142,-0.003049,0.057142,"[0.006606578826904297, -0.05129814147949219, 0...",0.001007,0.120525,0.9770
4,36,252,0.111675,0.010622,0.103708,"[0.018002748489379883, 0.07581663131713867, 0....",-0.004748,0.187529,0.9704
5,27,40,0.092108,-0.004402,0.092108,"[0.04875349998474121, 0.12630462646484375, -0....",-0.026714,0.175385,0.9286
6,33,1,0.090867,-0.025969,0.090867,"[-0.2565634250640869, 0.25287938117980957, -0....",-0.054175,0.246981,0.8784
7,27,146,0.232747,0.180549,0.097336,"[0.11967086791992188, -0.04862785339355469, 0....",-0.094028,0.315746,0.8226
8,33,166,0.048079,0.031907,0.024149,"[0.03655576705932617, 0.09870457649230957, 0.0...",-0.049989,0.086737,0.7428
9,33,169,0.042570,-0.003009,0.042570,"[0.0037183761596679688, 0.021992921829223633, ...",-0.022721,0.137710,0.7342


## 16 — Renormalization robustness

In [21]:
ROBUST_TOP_N = min(16, len(final_df))

robust_rows = []

for r in tqdm(
    list(final_df.head(ROBUST_TOP_N).itertuples(index=False)),
    desc="Renormalized validation",
):
    m = intervention_score(
        int(r.layer),
        [int(r.expert)],
        renormalize=True,
    )

    robust_rows.append({
        "layer": int(r.layer),
        "expert": int(r.expert),
        "renorm_target_delta_nll": m["target_delta_nll"],
        "renorm_control_delta_nll": m["control_delta_nll"],
        "renorm_causal_specificity": m["causal_specificity"],
    })

robust_df = pd.DataFrame(robust_rows)

final_robust = final_df.merge(
    robust_df,
    on=["layer","expert"],
    how="left",
)

final_robust.drop(columns=["per_example_delta"]).to_csv(
    RESULTS / "final_candidates_with_renorm.csv",
    index=False,
)

display(final_robust.head(20))

Renormalized validation: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:03<00:00,  4.58it/s]


,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,per_example_delta,ci_2.5,ci_97.5,p_positive,renorm_target_delta_nll,renorm_control_delta_nll,renorm_causal_specificity
0,36,229,1.285752,-0.031845,1.285752,"[1.2339627742767334, 1.6525044441223145, -0.09...",0.451026,2.197427,1.0000,1.306902,0.007985,1.300914
1,38,60,0.191670,0.002457,0.189827,"[0.3744649887084961, 0.48745298385620117, 0.08...",0.017480,0.355775,0.9858,0.197155,0.013401,0.187104
2,38,210,0.068980,0.000004,0.068977,"[-0.016318321228027344, -0.08301615715026855, ...",0.003809,0.150821,0.9810,0.082577,0.000004,0.082574
3,33,251,0.057142,-0.003049,0.057142,"[0.006606578826904297, -0.05129814147949219, 0...",0.001007,0.120525,0.9770,0.035986,-0.002417,0.035986
4,36,252,0.111675,0.010622,0.103708,"[0.018002748489379883, 0.07581663131713867, 0....",-0.004748,0.187529,0.9704,0.096877,0.000846,0.096243
5,27,40,0.092108,-0.004402,0.092108,"[0.04875349998474121, 0.12630462646484375, -0....",-0.026714,0.175385,0.9286,0.090335,-0.020521,0.090335
6,33,1,0.090867,-0.025969,0.090867,"[-0.2565634250640869, 0.25287938117980957, -0....",-0.054175,0.246981,0.8784,0.048686,-0.017878,0.048686
7,27,146,0.232747,0.180549,0.097336,"[0.11967086791992188, -0.04862785339355469, 0....",-0.094028,0.315746,0.8226,0.226403,0.271307,0.022922
8,33,166,0.048079,0.031907,0.024149,"[0.03655576705932617, 0.09870457649230957, 0.0...",-0.049989,0.086737,0.7428,0.042085,0.051791,0.003242
9,33,169,0.042570,-0.003009,0.042570,"[0.0037183761596679688, 0.021992921829223633, ...",-0.022721,0.137710,0.7342,0.042747,-0.001152,0.042747


## 17 — Routing diagnostic after causal selection

In [22]:
@contextmanager
def capture_routing():
    records = {}
    originals = []
    valid_flat = SCORE_BATCH["attention_mask"].reshape(-1).bool()

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(idx, original_forward):
            def patched(self, hidden_states):
                logits, weights, selected = original_forward(hidden_states)

                with torch.no_grad():
                    mask = valid_flat
                    if mask.numel() == selected.shape[0]:
                        mask = mask.to(selected.device)
                    else:
                        mask = torch.ones(
                            selected.shape[0],
                            device=selected.device,
                            dtype=torch.bool,
                        )

                    ids = selected[mask].reshape(-1).long()
                    ws = weights[mask].reshape(-1).float()

                    counts = torch.bincount(ids, minlength=cfg.num_experts)
                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )
                    wsum.scatter_add_(0, ids, ws)

                    records[int(idx)] = {
                        "tokens": int(mask.sum().item()),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return logits, weights, selected
            return patched

        gate.forward = types.MethodType(
            make_forward(layer_idx, original),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

with capture_routing() as routing_records:
    _ = score_cached_batch()

routing_rows = []

for layer_idx, rec in routing_records.items():
    tokens = max(1, rec["tokens"])

    for expert_id in range(cfg.num_experts):
        count = int(rec["counts"][expert_id].item())
        if count == 0:
            continue

        routing_rows.append({
            "layer": layer_idx,
            "expert": expert_id,
            "selected_rate": count / tokens,
            "routing_mass": float(rec["weight_sums"][expert_id].item()) / tokens,
        })

routing_df = pd.DataFrame(routing_rows)

comparison = final_robust.merge(
    routing_df,
    on=["layer","expert"],
    how="left",
).fillna({"selected_rate":0.0, "routing_mass":0.0})

comparison["causal_rank"] = comparison["causal_specificity"].rank(
    ascending=False, method="min"
)
comparison["routing_rank"] = comparison["routing_mass"].rank(
    ascending=False, method="min"
)
comparison["rank_gap"] = comparison["routing_rank"] - comparison["causal_rank"]

comparison.drop(columns=["per_example_delta"]).to_csv(
    RESULTS / "routing_vs_causality.csv",
    index=False,
)

display(
    comparison[
        [
            "layer","expert","causal_specificity","routing_mass",
            "causal_rank","routing_rank","rank_gap",
        ]
    ].sort_values("causal_rank").head(25)
)

,layer,expert,causal_specificity,routing_mass,causal_rank,routing_rank,rank_gap
0,36,229,1.285752,0.028932,1.0,5.0,4.0
1,38,60,0.189827,0.075386,2.0,1.0,-1.0
4,36,252,0.103708,0.061896,3.0,2.0,-1.0
7,27,146,0.097336,0.016141,4.0,7.0,3.0
5,27,40,0.092108,0.013331,5.0,9.0,4.0
6,33,1,0.090867,0.039948,6.0,3.0,-3.0
2,38,210,0.068977,0.020496,7.0,6.0,-1.0
3,33,251,0.057142,0.009347,8.0,12.0,4.0
9,33,169,0.042570,0.002843,9.0,21.0,12.0
10,27,69,0.033293,0.001031,10.0,28.0,18.0


## 18 — Optional same-layer expert-pair interactions

In [23]:
from itertools import combinations

RUN_PAIR_TESTS = True
PAIR_TOP_N = min(10, len(final_robust))

pair_rows = []

if RUN_PAIR_TESTS:
    top = final_robust.head(PAIR_TOP_N)

    for layer_idx, group in top.groupby("layer"):
        rows = list(group.itertuples(index=False))

        for a, b in combinations(rows, 2):
            m = intervention_score(
                int(layer_idx),
                [int(a.expert), int(b.expert)],
            )

            pair_rows.append({
                "layer": int(layer_idx),
                "expert_a": int(a.expert),
                "expert_b": int(b.expert),
                "pair_causal_specificity": m["causal_specificity"],
                "interaction_score": (
                    m["causal_specificity"]
                    - float(a.causal_specificity)
                    - float(b.causal_specificity)
                ),
            })

pair_df = pd.DataFrame(pair_rows)

if not pair_df.empty:
    pair_df = pair_df.sort_values(
        "pair_causal_specificity",
        ascending=False,
    )
    pair_df.to_csv(
        RESULTS / "same_layer_expert_interactions.csv",
        index=False,
    )
    display(pair_df)
else:
    print("No same-layer top-candidate pairs.")

,layer,expert_a,expert_b,pair_causal_specificity,interaction_score
7,36,229,252,1.417922,0.028462
8,38,60,210,0.269822,0.011017
5,33,1,169,0.132518,-0.000919
1,33,251,1,0.108767,-0.039242
3,33,251,169,0.102273,0.002560
4,33,1,166,0.096649,-0.018368
0,27,40,146,0.095129,-0.094315
6,33,166,169,0.092909,0.026189
2,33,251,166,0.062794,-0.018498


# Phase F — Selected-expert full-rank surgery

The full fused expert tensors remain frozen.

Only validated `(layer, expert)` slices are copied into a small external
trainable parameter bank.

The expert forward pass is monkeypatched only in layers that contain selected
experts:

- unselected expert → original frozen BF16 slice;
- selected expert → FP32 trainable master from the bank, autocast to BF16 for GEMM.

This avoids optimizer states for all 256 experts.

In [24]:
import torch.nn as nn

AUTO_SELECTED = (
    final_robust[
        (final_robust["p_positive"] >= 0.95)
        & (final_robust["causal_specificity"] > 0)
        & (
            final_robust["renorm_causal_specificity"].isna()
            | (final_robust["renorm_causal_specificity"] > 0)
        )
    ]
    .head(8)[["layer","expert"]]
)

SELECTED_EXPERTS = [
    (int(r.layer), int(r.expert))
    for r in AUTO_SELECTED.itertuples(index=False)
]

print("Automatic selected experts:", SELECTED_EXPERTS)

# Optional manual override:
# SELECTED_EXPERTS = [(18, 93), (22, 41)]

Automatic selected experts: [(36, 229), (38, 60), (38, 210), (33, 251), (36, 252)]


In [25]:
class SurgicalExpertBank(nn.Module):
    # Trainable FP32 copies of selected Laguna expert slices only.

    def __init__(self, selected_pairs):
        super().__init__()

        self.selected_pairs = sorted({
            (int(l), int(e))
            for l, e in selected_pairs
        })

        self.params = nn.ParameterDict()
        self.key_map = {}
        self.original_forwards = {}
        self.installed = False

        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(layer_idx).experts

            gu_key = f"L{layer_idx}_E{expert_id}_gu"
            down_key = f"L{layer_idx}_E{expert_id}_down"

            self.params[gu_key] = nn.Parameter(
                experts.gate_up_proj[expert_id].detach().float().clone()
            )
            self.params[down_key] = nn.Parameter(
                experts.down_proj[expert_id].detach().float().clone()
            )

            self.key_map[(layer_idx, expert_id)] = (gu_key, down_key)

        self.to("cuda:0")

    @property
    def trainable_parameter_count(self):
        return sum(p.numel() for p in self.parameters())

    def install(self):
        if self.installed:
            return

        for layer_idx in sorted({l for l, _ in self.selected_pairs}):
            experts = get_sparse_mlp(layer_idx).experts
            original = experts.forward
            self.original_forwards[layer_idx] = original

            selected_ids = {
                e for l, e in self.selected_pairs if l == layer_idx
            }
            bank = self

            def make_forward(idx, base_experts, selected):
                def surgical_forward(
                    self_experts,
                    hidden_states,
                    top_k_index,
                    top_k_weights,
                ):
                    final_hidden_states = torch.zeros_like(hidden_states)

                    with torch.no_grad():
                        expert_mask = F.one_hot(
                            top_k_index,
                            num_classes=self_experts.num_experts,
                        ).permute(2, 1, 0)

                        expert_hit = torch.greater(
                            expert_mask.sum(dim=(-1, -2)),
                            0,
                        ).nonzero(as_tuple=False).reshape(-1)

                    for expert_tensor in expert_hit:
                        expert_id = int(expert_tensor.item())
                        top_k_pos, token_idx = torch.where(
                            expert_mask[expert_id]
                        )
                        current_state = hidden_states[token_idx]

                        if expert_id in selected:
                            gu_key, down_key = bank.key_map[(idx, expert_id)]
                            gu = bank.params[gu_key].to(current_state.dtype)
                            down = bank.params[down_key].to(current_state.dtype)
                        else:
                            gu = base_experts.gate_up_proj[expert_id]
                            down = base_experts.down_proj[expert_id]

                        gate, up = F.linear(
                            current_state,
                            gu,
                        ).chunk(2, dim=-1)

                        h = base_experts.act_fn(gate) * up
                        h = F.linear(h, down)
                        h = h * top_k_weights[token_idx, top_k_pos, None]

                        final_hidden_states.index_add_(
                            0,
                            token_idx,
                            h.to(final_hidden_states.dtype),
                        )

                    return final_hidden_states

                return surgical_forward

            experts.forward = types.MethodType(
                make_forward(layer_idx, experts, selected_ids),
                experts,
            )

        self.installed = True

    def restore(self):
        if not self.installed:
            return

        for layer_idx, original in self.original_forwards.items():
            get_sparse_mlp(layer_idx).experts.forward = original

        self.original_forwards.clear()
        self.installed = False

    @torch.no_grad()
    def merge_into_base(self):
        for layer_idx, expert_id in self.selected_pairs:
            experts = get_sparse_mlp(layer_idx).experts
            gu_key, down_key = self.key_map[(layer_idx, expert_id)]

            experts.gate_up_proj[expert_id].copy_(
                self.params[gu_key].to(experts.gate_up_proj.dtype)
            )
            experts.down_proj[expert_id].copy_(
                self.params[down_key].to(experts.down_proj.dtype)
            )

## 19 — Surgical memory estimate

In [26]:
if SELECTED_EXPERTS:
    bank_preview = SurgicalExpertBank(SELECTED_EXPERTS)
    n = bank_preview.trainable_parameter_count

    print("Selected expert blocks:", len(SELECTED_EXPERTS))
    print(f"Trainable params: {n:,} ({n/1e6:.2f}M)")
    print(f"FP32 trainable bank: {n*4/2**30:.3f} GiB")
    print(f"Conservative params+grads+Adam states: {n*16/2**30:.3f} GiB")

    del bank_preview
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("No automatic candidates yet. Complete the causal search first.")

Selected expert blocks: 5
Trainable params: 15,728,640 (15.73M)
FP32 trainable bank: 0.059 GiB
Conservative params+grads+Adam states: 0.234 GiB


## 20 — Optional full-rank selected-expert training

Disabled by default.

Replace `TRAIN_ROWS` with a proper training split that is separate from causal
selection and held-out evaluation.

Recommended first run:

- 4–12 selected expert blocks
- microbatch 1
- sequence <=1024
- gradient accumulation 8–32
- AdamW
- learning rate 5e-6 to 2e-5
- gradient checkpointing enabled

In [27]:
TRAIN_ROWS = [
    {
        "prompt": "A flex child refuses to shrink and causes horizontal overflow. What CSS declaration should you try?",
        "reference": "min-width: 0;",
    },
    {
        "prompt": "In React, what hook is normally used to hold local state?",
        "reference": "useState",
    },
    {
        "prompt": "What CSS property creates spacing between children in flex and grid layouts?",
        "reference": "gap",
    },
]

RUN_SURGICAL_TRAINING = False
TRAIN_EPOCHS = 1
GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = 50
MAX_TRAIN_TOKENS = 1024

In [28]:
def make_training_case(prompt, reference, max_length=1024):
    prefix_text = chat_prefix_text(prompt)
    prefix_ids = tokenizer.encode(prefix_text, add_special_tokens=False)
    # Match the real assistant-message template: content follows
    # </think> on the next line.
    full_ids = tokenizer.encode(
        prefix_text + "\n" + reference,
        add_special_tokens=False,
    )

    if len(full_ids) > max_length:
        raise ValueError(f"{len(full_ids)} tokens > {max_length}")

    start = 0
    for a, b in zip(prefix_ids, full_ids):
        if a != b:
            break
        start += 1

    if start <= 0 or start >= len(full_ids):
        raise ValueError("Could not identify answer boundary.")

    input_ids = torch.tensor(
        full_ids,
        device="cuda:0",
        dtype=torch.long,
    ).unsqueeze(0)

    attention_mask = torch.ones_like(input_ids)

    pred_positions = torch.arange(
        start - 1,
        len(full_ids) - 1,
        device="cuda:0",
        dtype=torch.long,
    )

    targets = torch.tensor(
        full_ids[start:],
        device="cuda:0",
        dtype=torch.long,
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "pred_positions": pred_positions,
        "targets": targets,
    }

TRAIN_CASES = [
    make_training_case(
        r["prompt"],
        r["reference"],
        MAX_TRAIN_TOKENS,
    )
    for r in TRAIN_ROWS
]

print("Prepared training cases:", len(TRAIN_CASES))

Prepared training cases: 3


## 21 — Run training only when explicitly enabled

In [29]:
TRAIN_HISTORY = []

if RUN_SURGICAL_TRAINING:
    if not SELECTED_EXPERTS:
        raise RuntimeError("No selected experts.")

    bank = SurgicalExpertBank(SELECTED_EXPERTS)
    bank.install()
    bank.train()

    for p in model.parameters():
        p.requires_grad_(False)

    model.config.use_cache = False
    model.train()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=LEARNING_RATE,
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    raw_step = 0
    update_step = 0

    for epoch in range(TRAIN_EPOCHS):
        for case_idx, case in enumerate(TRAIN_CASES):
            raw_step += 1

            with torch.autocast("cuda", dtype=torch.bfloat16):
                out = model(
                    input_ids=case["input_ids"],
                    attention_mask=case["attention_mask"],
                    use_cache=False,
                    logits_to_keep=case["pred_positions"],
                    return_dict=True,
                )

                logits = out.logits.float()
                loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    case["targets"].reshape(-1),
                )
                scaled = loss / GRAD_ACCUM_STEPS

            scaled.backward()

            final_available_case = (
                epoch == TRAIN_EPOCHS - 1
                and case_idx == len(TRAIN_CASES) - 1
            )

            if (
                raw_step % GRAD_ACCUM_STEPS == 0
                or raw_step >= MAX_TRAIN_STEPS
                or final_available_case
            ):
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    bank.parameters(),
                    1.0,
                )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

                update_step += 1

                TRAIN_HISTORY.append({
                    "step": raw_step,
                    "update_step": update_step,
                    "loss": float(loss.detach().item()),
                    "grad_norm": float(grad_norm),
                })

                print(
                    f"step={raw_step} update={update_step} "
                    f"loss={loss.item():.4f} grad_norm={float(grad_norm):.3f}"
                )

            del out, logits, loss, scaled

            if raw_step >= MAX_TRAIN_STEPS:
                break

        if raw_step >= MAX_TRAIN_STEPS:
            break

    torch.cuda.synchronize()

    print(
        "Training peak allocated:",
        f"{torch.cuda.max_memory_allocated()/2**30:.2f} GiB",
    )
else:
    print("Training skipped.")

Training skipped.


## 22 — Post-training target/control check + compact save

In [30]:
if RUN_SURGICAL_TRAINING:
    model.eval()
    bank.eval()

    AFTER_TRAIN_NLL = score_cached_batch()
    post_delta = AFTER_TRAIN_NLL - BASE_NLL

    print(
        "Target mean ΔNLL after surgery:",
        float(post_delta[target_mask].mean()),
    )
    print(
        "Control mean ΔNLL after surgery:",
        float(post_delta[control_mask].mean()),
    )

    bank_path = RESULTS / "surgical_expert_bank.pt"

    torch.save(
        {
            "model_id": MODEL_ID,
            "selected_experts": SELECTED_EXPERTS,
            "state_dict": {
                k: v.detach().cpu()
                for k, v in bank.state_dict().items()
            },
            "train_history": TRAIN_HISTORY,
            "learning_rate": LEARNING_RATE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
        },
        bank_path,
    )

    print("Saved:", bank_path)
    print(f"Compact bank size: {bank_path.stat().st_size/2**20:.2f} MiB")

## 23 — Experiment manifest and results archive

In [31]:
import json, shutil
from datetime import datetime, timezone

free_b, total_b = torch.cuda.mem_get_info()

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "model_path": str(MODEL_PATH),
    "backend": "PyTorch/Transformers BF16",
    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "gpu_vram_gib": torch.cuda.get_device_properties(0).total_memory/2**30,
        "logical_cpus": os.cpu_count(),
        "ram_gib": psutil.virtual_memory().total/2**30,
    },
    "software": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "cuda": torch.version.cuda,
    },
    "architecture": {
        "hidden_size": int(cfg.hidden_size),
        "layers": int(cfg.num_hidden_layers),
        "sparse_layers": list(map(int, SPARSE_LAYERS)),
        "experts": int(cfg.num_experts),
        "top_k": int(cfg.num_experts_per_tok),
        "expert_width": int(cfg.moe_intermediate_size),
        "params_per_expert": int(params_per_expert),
    },
    "search": {
        "control_penalty": CONTROL_PENALTY,
        "top_layers": TOP_LAYERS,
        "search_seeds": SEARCH_SEEDS,
        "initial_group_size": INITIAL_GROUP_SIZE,
        "beam_width": BEAM_WIDTH,
        "eval_examples": len(eval_df),
    },
    "selected_experts": SELECTED_EXPERTS,
    "training_enabled": RUN_SURGICAL_TRAINING,
    "gpu_end": {
        "allocated_gib": torch.cuda.memory_allocated()/2**30,
        "reserved_gib": torch.cuda.memory_reserved()/2**30,
        "free_gib": free_b/2**30,
        "peak_gib": torch.cuda.max_memory_allocated()/2**30,
    },
}

(RESULTS / "manifest.json").write_text(json.dumps(manifest, indent=2))

archive = shutil.make_archive(
    str(RESULTS),
    "zip",
    root_dir=RESULTS,
)

print("Results:", RESULTS)
print("Archive:", archive)

display(
    final_robust[
        [
            "layer","expert",
            "target_delta_nll",
            "control_delta_nll",
            "causal_specificity",
            "renorm_causal_specificity",
            "ci_2.5","ci_97.5",
            "p_positive",
        ]
    ].head(20)
)

Results: /home/ec2-user/workspace/laguna_xs2_causal_surgery_results
Archive: /home/ec2-user/workspace/laguna_xs2_causal_surgery_results.zip


,layer,expert,target_delta_nll,control_delta_nll,causal_specificity,renorm_causal_specificity,ci_2.5,ci_97.5,p_positive
0,36,229,1.285752,-0.031845,1.285752,1.300914,0.451026,2.197427,1.0000
1,38,60,0.191670,0.002457,0.189827,0.187104,0.017480,0.355775,0.9858
2,38,210,0.068980,0.000004,0.068977,0.082574,0.003809,0.150821,0.9810
3,33,251,0.057142,-0.003049,0.057142,0.035986,0.001007,0.120525,0.9770
4,36,252,0.111675,0.010622,0.103708,0.096243,-0.004748,0.187529,0.9704
5,27,40,0.092108,-0.004402,0.092108,0.090335,-0.026714,0.175385,0.9286
6,33,1,0.090867,-0.025969,0.090867,0.048686,-0.054175,0.246981,0.8784
7,27,146,0.232747,0.180549,0.097336,0.022922,-0.094028,0.315746,0.8226
8,33,166,0.048079,0.031907,0.024149,0.003242,-0.049989,0.086737,0.7428
9,33,169,0.042570,-0.003009,0.042570,0.042747,-0.022721,0.137710,0.7342


# Publication-quality protocol

Before claiming a result:

1. replace the toy probe set with a proper selection dataset;
2. use a disjoint held-out target/control evaluation set;
3. repeat expert search with multiple randomized group orderings;
4. compare matched trainable-parameter/data budgets:
   - random experts,
   - most-routed experts,
   - gradient/saliency,
   - task/success association,
   - causal expert surgery;
5. report target gain, general retention, forgetting, parameter count,
   optimizer memory, search compute, and routing-vs-causal rank correlation;
6. test both intervention semantics and a small number of coalitions.

The falsifiable question is:

> At the same parameter and data budget, does measured causal contribution
> select a better sparse adaptation circuit than observational expert-selection
> methods?

# Phase H — Publication-oriented matched-selector experiment

This phase turns the causal-localization result into a falsifiable adaptation
comparison.

The default **one-expert parameter budget** is identical across arms:

1. **causal** — strongest expert with a positive bootstrap lower bound and
   positive renormalized causal score;
2. **routing** — globally highest routing-mass expert on the target selection
   split;
3. **random-global** — one uniformly random `(layer, expert)` pair;
4. **random-same-layer** — one random expert from the causal expert's layer.

Each arm:

- starts from the same frozen BF16 Laguna backbone;
- trains exactly one full-rank expert block;
- sees the exact same target training examples;
- uses the same optimizer, LR, update count and gradient accumulation;
- is evaluated on the same held-out target/control set.

This is the cleanest first test of:

\[
\text{causal selection} \stackrel{?}{>} \text{routing selection / random}
\]

at a matched ~3.146M-parameter budget.

## External dataset format

For a serious run, set:

```bash
export LAGUNA_EXPERIMENT_CSV=/path/to/laguna_experiment.csv
```

CSV columns:

```text
split,kind,prompt,reference
selection,target,...
selection,control,...
train,target,...
heldout,target,...
heldout,control,...
```

Recommended minimum:

- selection: 50+ target and 50+ control;
- train: 50+ target;
- heldout: 50+ target and 50+ control.

The notebook will refuse a publication-mode run on a tiny dataset unless you
explicitly set `ALLOW_SMALL_DATASET=True`.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

EXPERIMENT_CSV = os.environ.get("LAGUNA_EXPERIMENT_CSV", "").strip()
ALLOW_SMALL_DATASET = False

if EXPERIMENT_CSV:
    experiment_df = pd.read_csv(EXPERIMENT_CSV)

    required_cols = {"split", "kind", "prompt", "reference"}
    missing_cols = required_cols - set(experiment_df.columns)
    if missing_cols:
        raise ValueError(f"Dataset missing required columns: {sorted(missing_cols)}")

    experiment_df["split"] = experiment_df["split"].astype(str).str.lower()
    experiment_df["kind"] = experiment_df["kind"].astype(str).str.lower()

    allowed_splits = {"selection", "train", "heldout"}
    allowed_kinds = {"target", "control"}

    bad_splits = sorted(set(experiment_df["split"]) - allowed_splits)
    bad_kinds = sorted(set(experiment_df["kind"]) - allowed_kinds)

    if bad_splits:
        raise ValueError(f"Unsupported split labels: {bad_splits}")
    if bad_kinds:
        raise ValueError(f"Unsupported kind labels: {bad_kinds}")

    counts = (
        experiment_df.groupby(["split", "kind"])
        .size()
        .rename("count")
        .reset_index()
    )
    display(counts)

    required_minima = {
        ("selection", "target"): 50,
        ("selection", "control"): 50,
        ("train", "target"): 50,
        ("heldout", "target"): 50,
        ("heldout", "control"): 50,
    }

    too_small = []
    for key, minimum in required_minima.items():
        actual = len(
            experiment_df[
                (experiment_df["split"] == key[0])
                & (experiment_df["kind"] == key[1])
            ]
        )
        if actual < minimum:
            too_small.append((key, actual, minimum))

    if too_small and not ALLOW_SMALL_DATASET:
        raise RuntimeError(
            "Dataset is too small for publication mode:\n"
            + "\n".join(
                f"{split}/{kind}: {actual} < {minimum}"
                for (split, kind), actual, minimum in too_small
            )
            + "\nSet ALLOW_SMALL_DATASET=True only for a pipeline smoke test."
        )

    PUBLICATION_DATA = True

else:
    # Fall back to the already-run small probe set only for pipeline validation.
    PUBLICATION_DATA = False

    experiment_df = eval_df.copy()
    experiment_df["split"] = "selection"

    # Reuse target probes as a tiny training smoke set.
    smoke_train = eval_df[eval_df["kind"] == "target"].copy()
    smoke_train["split"] = "train"

    # Reuse evaluation probes as heldout only so the code path is runnable.
    # This is NOT scientifically valid and is labeled accordingly.
    smoke_heldout = eval_df.copy()
    smoke_heldout["split"] = "heldout"

    experiment_df = pd.concat(
        [experiment_df, smoke_train, smoke_heldout],
        ignore_index=True,
    )

    print(
        "WARNING: LAGUNA_EXPERIMENT_CSV is not set. "
        "Using the tiny built-in probes only to validate the matched-baseline pipeline."
    )

print("PUBLICATION_DATA =", PUBLICATION_DATA)
display(
    experiment_df.groupby(["split", "kind"]).size().rename("count")
)

## H1 — Generic scoring-batch builder for arbitrary splits

This uses the same corrected Laguna assistant formatting as the causal atlas:

```text
<assistant>
</think>
REFERENCE
```

In [ ]:
def build_scoring_batch_from_df(df):
    df = df.reset_index(drop=True).copy()

    parsed = [parse_case(r.prompt, r.reference) for r in df.itertuples(index=False)]
    fulls = [x[0] for x in parsed]
    starts = [x[1] for x in parsed]
    answers = [full[start:] for full, start in zip(fulls, starts)]

    max_prefix = max(starts)
    max_ref = max(len(x) for x in answers)
    batch_size = len(df)
    seq_len = max_prefix + max_ref

    input_ids = torch.full(
        (batch_size, seq_len),
        tokenizer.pad_token_id,
        dtype=torch.long,
    )
    attention_mask = torch.zeros((batch_size, seq_len), dtype=torch.long)
    targets = torch.full((batch_size, max_ref), -100, dtype=torch.long)

    for b, (full, start, ans) in enumerate(zip(fulls, starts, answers)):
        prefix = full[:start]
        prefix_start = max_prefix - len(prefix)

        input_ids[b, prefix_start:max_prefix] = torch.tensor(prefix)
        attention_mask[b, prefix_start:max_prefix] = 1

        input_ids[b, max_prefix:max_prefix+len(ans)] = torch.tensor(ans)
        attention_mask[b, max_prefix:max_prefix+len(ans)] = 1
        targets[b, :len(ans)] = torch.tensor(ans)

    position_ids = attention_mask.cumsum(dim=-1) - 1
    position_ids.clamp_(min=0)

    pred_positions = torch.arange(
        max_prefix - 1,
        max_prefix + max_ref - 1,
        dtype=torch.long,
    )

    return {
        "df": df,
        "input_ids": input_ids.to("cuda:0", non_blocking=True),
        "attention_mask": attention_mask.to("cuda:0", non_blocking=True),
        "position_ids": position_ids.to("cuda:0", non_blocking=True),
        "targets": targets.to("cuda:0", non_blocking=True),
        "pred_positions": pred_positions.to("cuda:0", non_blocking=True),
    }

@torch.inference_mode()
def score_batch_object(batch):
    out = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        position_ids=batch["position_ids"],
        use_cache=False,
        output_router_logits=False,
        logits_to_keep=batch["pred_positions"],
        return_dict=True,
    )

    logits = out.logits.float()
    targets = batch["targets"]

    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        ignore_index=-100,
        reduction="none",
    ).reshape(targets.shape)

    valid = targets.ne(-100)
    per_example = (losses * valid).sum(-1) / valid.sum(-1).clamp_min(1)

    result = per_example.detach().cpu().numpy()

    del out, logits, losses, per_example
    return result

selection_df = experiment_df[experiment_df["split"] == "selection"].reset_index(drop=True)
train_target_df = experiment_df[
    (experiment_df["split"] == "train")
    & (experiment_df["kind"] == "target")
].reset_index(drop=True)
heldout_df = experiment_df[experiment_df["split"] == "heldout"].reset_index(drop=True)

SELECTION_BATCH = build_scoring_batch_from_df(selection_df)
HELDOUT_BATCH = build_scoring_batch_from_df(heldout_df)

HELDOUT_BASE_NLL = score_batch_object(HELDOUT_BATCH)

heldout_target_mask = heldout_df["kind"].values == "target"
heldout_control_mask = heldout_df["kind"].values == "control"

print("Heldout baseline target NLL :", float(HELDOUT_BASE_NLL[heldout_target_mask].mean()))
print("Heldout baseline control NLL:", float(HELDOUT_BASE_NLL[heldout_control_mask].mean()))

## H2 — Strict causal selector

The previous permissive shortlist allowed `p_positive >= 0.95` even if the
95% bootstrap interval barely crossed zero.

For the matched one-expert experiment, use a stricter rule:

- `ci_2.5 > 0`;
- `causal_specificity > 0`;
- `renorm_causal_specificity > 0`.

Then choose the highest causal-specificity expert.

In [ ]:
strict_candidates = final_robust[
    (final_robust["ci_2.5"] > 0)
    & (final_robust["causal_specificity"] > 0)
    & (final_robust["renorm_causal_specificity"] > 0)
].copy()

if strict_candidates.empty:
    raise RuntimeError(
        "No expert passed the strict causal criterion. "
        "Do not force a causal arm."
    )

strict_candidates = strict_candidates.sort_values(
    "causal_specificity",
    ascending=False,
).reset_index(drop=True)

CAUSAL_PAIR = (
    int(strict_candidates.iloc[0]["layer"]),
    int(strict_candidates.iloc[0]["expert"]),
)

print("Strict causal candidates:")
display(
    strict_candidates[
        [
            "layer", "expert",
            "causal_specificity",
            "ci_2.5", "ci_97.5",
            "renorm_causal_specificity",
            "routing_mass",
        ]
        if "routing_mass" in strict_candidates.columns
        else [
            "layer", "expert",
            "causal_specificity",
            "ci_2.5", "ci_97.5",
            "renorm_causal_specificity",
        ]
    ].head(10)
)

print("CAUSAL_PAIR =", CAUSAL_PAIR)

## H3 — Measure global target-selection routing activity

Unlike the earlier candidate-only routing table, this pass ranks **all**
39 × 256 layer/expert positions on the selection split.

This gives a fair observational routing baseline.

In [ ]:
from contextlib import contextmanager
import types

@contextmanager
def capture_routing_for_batch(batch):
    records = {}
    originals = []

    valid_flat = batch["attention_mask"].reshape(-1).bool()

    for layer_idx in SPARSE_LAYERS:
        gate = get_sparse_mlp(layer_idx).gate
        original = gate.forward
        originals.append((gate, original))

        def make_forward(idx, original_forward):
            def patched(self, hidden_states):
                logits, weights, selected = original_forward(hidden_states)

                with torch.no_grad():
                    mask = valid_flat
                    if mask.numel() == selected.shape[0]:
                        mask = mask.to(selected.device)
                    else:
                        mask = torch.ones(
                            selected.shape[0],
                            device=selected.device,
                            dtype=torch.bool,
                        )

                    ids = selected[mask].reshape(-1).long()
                    ws = weights[mask].reshape(-1).float()

                    counts = torch.bincount(
                        ids,
                        minlength=cfg.num_experts,
                    )

                    wsum = torch.zeros(
                        cfg.num_experts,
                        device=ws.device,
                        dtype=torch.float32,
                    )
                    wsum.scatter_add_(0, ids, ws)

                    records[int(idx)] = {
                        "tokens": int(mask.sum().item()),
                        "counts": counts.cpu(),
                        "weight_sums": wsum.cpu(),
                    }

                return logits, weights, selected

            return patched

        gate.forward = types.MethodType(
            make_forward(layer_idx, original),
            gate,
        )

    try:
        yield records
    finally:
        for gate, original in originals:
            gate.forward = original

# Routing baseline should be selected using target-selection examples only.
selection_target_df = selection_df[selection_df["kind"] == "target"].reset_index(drop=True)
SELECTION_TARGET_BATCH = build_scoring_batch_from_df(selection_target_df)

with capture_routing_for_batch(SELECTION_TARGET_BATCH) as global_routing_records:
    _ = score_batch_object(SELECTION_TARGET_BATCH)

global_routing_rows = []

for layer_idx in SPARSE_LAYERS:
    rec = global_routing_records[int(layer_idx)]
    tokens = max(1, rec["tokens"])

    for expert_id in range(cfg.num_experts):
        global_routing_rows.append({
            "layer": int(layer_idx),
            "expert": int(expert_id),
            "selected_rate": float(rec["counts"][expert_id].item()) / tokens,
            "routing_mass": float(rec["weight_sums"][expert_id].item()) / tokens,
        })

global_routing_df = pd.DataFrame(global_routing_rows).sort_values(
    ["routing_mass", "selected_rate"],
    ascending=False,
).reset_index(drop=True)

ROUTING_PAIR = (
    int(global_routing_df.iloc[0]["layer"]),
    int(global_routing_df.iloc[0]["expert"]),
)

print("Global most-routed target-selection expert:", ROUTING_PAIR)

display(global_routing_df.head(20))

global_routing_df.to_csv(
    RESULTS / "global_target_selection_routing.csv",
    index=False,
)

## H4 — Construct matched one-expert selector arms

All arms train exactly one 3.146M-parameter routed expert block.

In [ ]:
MATCHED_RANDOM_SEED = 2026
rng = np.random.default_rng(MATCHED_RANDOM_SEED)

all_pairs = [
    (int(layer_idx), int(expert_id))
    for layer_idx in SPARSE_LAYERS
    for expert_id in range(cfg.num_experts)
]

RANDOM_GLOBAL_PAIR = all_pairs[
    int(rng.integers(0, len(all_pairs)))
]

same_layer_choices = [
    (CAUSAL_PAIR[0], e)
    for e in range(cfg.num_experts)
    if e != CAUSAL_PAIR[1]
]
RANDOM_SAME_LAYER_PAIR = same_layer_choices[
    int(rng.integers(0, len(same_layer_choices)))
]

SELECTOR_ARMS = {
    "causal": CAUSAL_PAIR,
    "routing": ROUTING_PAIR,
    "random_global": RANDOM_GLOBAL_PAIR,
    "random_same_layer": RANDOM_SAME_LAYER_PAIR,
}

selector_df = pd.DataFrame(
    [
        {"selector": name, "layer": pair[0], "expert": pair[1]}
        for name, pair in SELECTOR_ARMS.items()
    ]
)

display(selector_df)

print(
    "All arms have identical trainable parameter count:",
    f"{params_per_expert:,} params",
)

## H5 — Build identical target-training cases

Training is reference-NLL adaptation with one example per microbatch.

The exact same ordered examples are used for every selector arm.

In [ ]:
MATCHED_MAX_TRAIN_TOKENS = 1024

MATCHED_TRAIN_CASES = [
    make_training_case(
        r.prompt,
        r.reference,
        max_length=MATCHED_MAX_TRAIN_TOKENS,
    )
    for r in train_target_df.itertuples(index=False)
]

print("Matched training examples:", len(MATCHED_TRAIN_CASES))
print(
    "Token length range:",
    min(x["input_ids"].shape[1] for x in MATCHED_TRAIN_CASES),
    "to",
    max(x["input_ids"].shape[1] for x in MATCHED_TRAIN_CASES),
)

## H6 — Matched training/evaluation function

Important properties:

- each arm creates a brand-new `SurgicalExpertBank`;
- no trained weights are merged into the frozen base;
- `bank.restore()` reinstates the original expert forward before the next arm;
- optimizer state is discarded between arms;
- training order, LR, update budget and accumulation are identical.

In [ ]:
MATCHED_LR = 1e-5
MATCHED_GRAD_ACCUM = 8
MATCHED_EPOCHS = 3
MATCHED_MAX_UPDATES = 50
MATCHED_WEIGHT_DECAY = 0.01

def evaluate_heldout_with_current_model():
    nll = score_batch_object(HELDOUT_BATCH)

    target_mean = float(nll[heldout_target_mask].mean())
    control_mean = float(nll[heldout_control_mask].mean())

    target_improvement = float(
        HELDOUT_BASE_NLL[heldout_target_mask].mean() - target_mean
    )
    control_damage = float(
        control_mean - HELDOUT_BASE_NLL[heldout_control_mask].mean()
    )

    adaptation_score = (
        target_improvement
        - CONTROL_PENALTY * max(control_damage, 0.0)
    )

    return {
        "heldout_target_nll": target_mean,
        "heldout_control_nll": control_mean,
        "target_improvement": target_improvement,
        "control_damage": control_damage,
        "adaptation_score": adaptation_score,
        "per_example_nll": nll,
    }

def train_one_selector_arm(selector_name, pair):
    layer_idx, expert_id = map(int, pair)

    bank = SurgicalExpertBank([(layer_idx, expert_id)])
    bank.install()
    bank.train()
    model.train()
    model.config.use_cache = False

    for p in model.parameters():
        p.requires_grad_(False)

    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    try:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model.gradient_checkpointing_enable()

    optimizer = torch.optim.AdamW(
        bank.parameters(),
        lr=MATCHED_LR,
        betas=(0.9, 0.95),
        weight_decay=MATCHED_WEIGHT_DECAY,
    )

    optimizer.zero_grad(set_to_none=True)

    history = []
    raw_step = 0
    update_step = 0

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    try:
        for epoch in range(MATCHED_EPOCHS):
            for case_idx, case in enumerate(MATCHED_TRAIN_CASES):
                raw_step += 1

                with torch.autocast("cuda", dtype=torch.bfloat16):
                    out = model(
                        input_ids=case["input_ids"],
                        attention_mask=case["attention_mask"],
                        use_cache=False,
                        logits_to_keep=case["pred_positions"],
                        return_dict=True,
                    )

                    logits = out.logits.float()

                    loss = F.cross_entropy(
                        logits.reshape(-1, logits.shape[-1]),
                        case["targets"].reshape(-1),
                    )

                    scaled = loss / MATCHED_GRAD_ACCUM

                scaled.backward()

                last_available_case = (
                    epoch == MATCHED_EPOCHS - 1
                    and case_idx == len(MATCHED_TRAIN_CASES) - 1
                )

                if (
                    raw_step % MATCHED_GRAD_ACCUM == 0
                    or last_available_case
                ):
                    grad_norm = torch.nn.utils.clip_grad_norm_(
                        bank.parameters(),
                        1.0,
                    )

                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    update_step += 1

                    history.append({
                        "selector": selector_name,
                        "layer": layer_idx,
                        "expert": expert_id,
                        "raw_step": raw_step,
                        "update_step": update_step,
                        "loss": float(loss.detach().item()),
                        "grad_norm": float(grad_norm),
                    })

                del out, logits, loss, scaled

                if update_step >= MATCHED_MAX_UPDATES:
                    break

            if update_step >= MATCHED_MAX_UPDATES:
                break

        model.eval()
        bank.eval()

        metrics = evaluate_heldout_with_current_model()

        bank_state_cpu = {
            k: v.detach().cpu().clone()
            for k, v in bank.state_dict().items()
        }

        peak_gib = torch.cuda.max_memory_allocated() / 2**30

        result = {
            "selector": selector_name,
            "layer": layer_idx,
            "expert": expert_id,
            "trainable_params": bank.trainable_parameter_count,
            "updates": update_step,
            "final_train_loss": (
                history[-1]["loss"] if history else np.nan
            ),
            "peak_gpu_gib": peak_gib,
            "heldout_target_nll": metrics["heldout_target_nll"],
            "heldout_control_nll": metrics["heldout_control_nll"],
            "target_improvement": metrics["target_improvement"],
            "control_damage": metrics["control_damage"],
            "adaptation_score": metrics["adaptation_score"],
            "history": history,
            "bank_state_cpu": bank_state_cpu,
            "per_example_nll": metrics["per_example_nll"],
        }

        return result

    finally:
        bank.restore()

        if hasattr(model, "disable_input_require_grads"):
            model.disable_input_require_grads()

        try:
            model.gradient_checkpointing_disable()
        except Exception:
            pass

        model.eval()

        del optimizer
        del bank
        gc.collect()
        torch.cuda.empty_cache()

## H7 — Run the matched experiment

Disabled by default so merely opening/running the notebook cannot accidentally
start training.

After checking the dataset counts and selector arms:

```python
RUN_MATCHED_EXPERIMENT = True
```

In [ ]:
RUN_MATCHED_EXPERIMENT = False

MATCHED_RESULTS = {}
MATCHED_RESULT_ROWS = []
MATCHED_HISTORY_ROWS = []

if RUN_MATCHED_EXPERIMENT:
    if not PUBLICATION_DATA:
        print(
            "WARNING: running in smoke mode on reused tiny probes. "
            "Do not interpret these numbers scientifically."
        )

    for selector_name, pair in SELECTOR_ARMS.items():
        print("\n" + "=" * 72)
        print("Training selector:", selector_name, "pair:", pair)
        print("=" * 72)

        result = train_one_selector_arm(selector_name, pair)
        MATCHED_RESULTS[selector_name] = result

        MATCHED_RESULT_ROWS.append({
            k: v
            for k, v in result.items()
            if k not in {
                "history",
                "bank_state_cpu",
                "per_example_nll",
            }
        })

        MATCHED_HISTORY_ROWS.extend(result["history"])

        print(
            f"{selector_name}: "
            f"target improvement={result['target_improvement']:+.4f}, "
            f"control damage={result['control_damage']:+.4f}, "
            f"adaptation score={result['adaptation_score']:+.4f}"
        )

    matched_results_df = pd.DataFrame(MATCHED_RESULT_ROWS).sort_values(
        "adaptation_score",
        ascending=False,
    ).reset_index(drop=True)

    matched_history_df = pd.DataFrame(MATCHED_HISTORY_ROWS)

    display(matched_results_df)

else:
    print("Matched training is disabled.")

## H8 — Save matched experiment artifacts

Each selector's trained bank is tiny: one full Laguna expert block only.

In [ ]:
if RUN_MATCHED_EXPERIMENT:
    MATCHED_DIR = RESULTS / "matched_one_expert_experiment"
    MATCHED_DIR.mkdir(parents=True, exist_ok=True)

    matched_results_df.to_csv(
        MATCHED_DIR / "matched_selector_results.csv",
        index=False,
    )

    matched_history_df.to_csv(
        MATCHED_DIR / "matched_training_history.csv",
        index=False,
    )

    selector_df.to_csv(
        MATCHED_DIR / "selector_arms.csv",
        index=False,
    )

    for selector_name, result in MATCHED_RESULTS.items():
        torch.save(
            {
                "model_id": MODEL_ID,
                "selector": selector_name,
                "layer": result["layer"],
                "expert": result["expert"],
                "trainable_params": result["trainable_params"],
                "learning_rate": MATCHED_LR,
                "grad_accum": MATCHED_GRAD_ACCUM,
                "updates": result["updates"],
                "state_dict": result["bank_state_cpu"],
            },
            MATCHED_DIR / f"{selector_name}_expert_bank.pt",
        )

        np.save(
            MATCHED_DIR / f"{selector_name}_heldout_nll.npy",
            result["per_example_nll"],
        )

    print("Saved matched experiment to:", MATCHED_DIR)

# Recommended interpretation

The strongest first paper-quality claim would require all of the following:

1. the same causal expert remains strong on a larger **selection** set;
2. its bootstrap lower bound remains above zero;
3. its effect survives routing-weight renormalization;
4. on a disjoint **held-out** set, training the causal expert beats:
   - the globally most-routed expert,
   - a random global expert,
   - a random same-layer expert;
5. all arms use exactly the same parameter count, examples, optimizer,
   learning rate and update budget.

For the one-expert experiment the trainable parameter budget is only about
3.146M parameters, roughly 0.01% of the 33B-total model.